#
**Task 1 solution**
***

##
**Importing packages and data**
***

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import poisson

df = pd.read_csv("poisson_params_Quatar_2022.csv")

print(df.columns)
df

Index(['team', 'alpha', 'beta', 'gamma', 'delta'], dtype='object')


,team,alpha,beta,gamma,delta
0,ALB,0.430117,2.158279,0.708363,1.020914
1,ALG,0.692026,1.961996,0.758433,1.908582
2,ARG,1.000867,1.213104,0.711123,0.485013
3,AUS,0.586533,2.202754,0.893709,0.828078
4,BEL,1.102921,1.409477,1.381494,1.197060
...,...,...,...,...,...
66,URU,0.702663,1.451352,1.409962,0.813889
67,USA,0.650696,1.846081,1.180708,0.938874
68,UZB,0.311460,3.254165,1.487716,0.558699
69,VEN,0.539220,2.216620,1.321555,0.985907


In [16]:
#Set team as index and convert to dictionary to access parameters for each team 
params = df.set_index("team").to_dict(orient = "index")
params

{'ALB': {'alpha': 0.4301171963037186,
  'beta': 2.1582786733032555,
  'gamma': 0.7083627782525629,
  'delta': 1.020914264753698},
 'ALG': {'alpha': 0.6920261594648714,
  'beta': 1.9619964259128992,
  'gamma': 0.7584325986235402,
  'delta': 1.9085816227181904},
 'ARG': {'alpha': 1.00086685345433,
  'beta': 1.2131036064788594,
  'gamma': 0.7111234523879356,
  'delta': 0.4850131568362615},
 'AUS': {'alpha': 0.586533407806349,
  'beta': 2.202753679826695,
  'gamma': 0.8937092488115446,
  'delta': 0.8280781537790579},
 'BEL': {'alpha': 1.1029207842817736,
  'beta': 1.4094772126183,
  'gamma': 1.381494049271062,
  'delta': 1.197060077735404},
 'BKF': {'alpha': 0.4032906419994814,
  'beta': 2.5022140253831635,
  'gamma': 1.6522340534144813,
  'delta': 0.9521317939619346},
 'BOL': {'alpha': 0.3722842790650112,
  'beta': 3.6007101338554945,
  'gamma': 2.799112869513658,
  'delta': 0.5732371092334628},
 'BOS': {'alpha': 0.4891846193761389,
  'beta': 1.7331537275342552,
  'gamma': 1.3485667637007

##
**Calculating expected goals**
***

In [3]:
def expected_goals(home_team, away_team, params, home_effect = False):

    """
    Calculate expected goals for home and away team using specified parameters
    
    Parameters:
        home_team (str): Name of the home team
        away_team (str): Name of the away team
        params (dict): Dictionary containing the parameters for each team:
            alpha, beta, gamma, delta
        home_effect (bool): Whether to include a home effect in the calculation. Default is False.
    
    Returns:
        tuple(float, float): Expected goals for home and away team
    """
    
    #Offensive and defensive parameters for home and away team
    alpha_i = params[home_team]["alpha"]
    beta_i = params[home_team]["beta"]
    gamma_i = params[home_team]["gamma"]
    delta_i = params[home_team]["delta"]

    alpha_j = params[away_team]["alpha"]
    beta_j = params[away_team]["beta"]
    gamma_j = params[away_team]["gamma"]
    delta_j = params[away_team]["delta"]

    #Calulate expected goals without home effect
    lam = alpha_i * beta_j
    mu = alpha_j * beta_i

    #Calculate expected goals with home effect
    if home_effect:

        lam = alpha_i * beta_j * gamma_i
        mu = alpha_j * beta_i * delta_j

    return lam, mu

##
**Group stage**
***

In [4]:
groups = {
    "A" : ["NET", "SEN", "ECU", "QAT"],
    "B" : ["ENG", "USA", "IRN", "WAL"],
    "C" : ["ARG", "KSA", "MEX", "POL"],
    "D" : ["FRA", "AUS", "DEN", "TUN"],
    "E" : ["ESP", "CRC", "GER", "JAP"],
    "F" : ["BEL", "CAN", "MAR", "CRO"],
    "G" : ["BRA", "SRB", "SUE", "CMR"],
    "H" : ["POR", "GHA", "URU", "KOR"],
}

In [5]:
#Generating all fixtures for each group
def fixtures(teams):

    """
    Generate round-robin fixtures for each group

    Parameters:
        teams (list[str]): List of teams in the group

    Returns:
        list [tuple[str, str]]: List of tuples with all fixtures in the group
    """
    
    return [
        (teams[0], teams[1]),
        (teams[0], teams[2]),
        (teams[0], teams[3]),
        (teams[1], teams[2]),
        (teams[1], teams[3]),
        (teams[2], teams[3]),
    ]

In [6]:
#Store table statistics for each team in a dictionary
def table(teams):
    """
    Initialize empty table for each team

    Parameters:
        teams (list[str]): List of teams in the group

    Returns: 
        dict: Dictionary with statsitics, initialized at 0 for each team
    """
    
    table = {}

    for team in teams:
        table[team] = {
            "Points" : 0,
            "Goals For" : 0,
            "Goals Against" : 0,
            "Goal Difference" : 0,
        }
    return table

##
**Updating and sorting table**
***

In [7]:
def update_table(table, home_team, away_team, home_goals, away_goals):
    
    """
    Update the table with results after a match

    Parameters:
        table (dict): Table with team statistics
        home_team (str): Home team
        away_team (str): Away team
        home_goals (int): Goals scored by the home team
        away_goals (int): Goals scored by the away team

    Returns:
        None: The function updates existing table and doesn't return anything
    """

    #Update goals for and against
    table[home_team]["Goals For"] += home_goals
    table[home_team]["Goals Against"] += away_goals

    table[away_team]["Goals For"] += away_goals
    table[away_team]["Goals Against"] += home_goals

    #Update goal difference
    table[home_team]["Goal Difference"] = table[home_team]["Goals For"] - table[home_team]["Goals Against"]
    table[away_team]["Goal Difference"] = table[away_team]["Goals For"] - table[away_team]["Goals Against"]

    #3 points if home_team wins
    if home_goals > away_goals:
        table[home_team]["Points"] += 3

    #3 points if away_team wins
    elif home_goals < away_goals:
        table[away_team]["Points"] += 3

    #1 point each if they draw
    else:
        table[home_team]["Points"] += 1
        table[away_team]["Points"] += 1
    
    #0 points if they lose, so no need to update for that case

In [8]:
def sort_table(group_table):
    
    """
    Sort the group table based on stats

    Parameters:
        group_table (dict): Table with team statistics

    Returns:
        sorted_table (dict): Sorted table with team statistics
    """
    
    sorted_table = sorted( 
        group_table.items(), #Convert dictionary into (team, stats) tuples for iteration and indexing
        key = lambda x: ( #For every item, return a tuple of stats in this particular order
            x[1]["Points"],
            x[1]["Goal Difference"], #Index 1 to access the second item in the tuple which is the stats dictionary
            x[1]["Goals For"],
        ),

        reverse = True, #Sort in descending order
    )

    return sorted_table

##
**Computing before simulation**
***

In [9]:
#Compute results and their probabilities once before the monte carlo simulation, as this is the most consuming part of the simulation, and we can reuse the results for each simulation
def compute_matches(groups):

    """
    Compute scores for all matches in group stages using given parameters

    Parameters:
        groups (dict): Dictionary with group names as keys and list of teams as values

    Returns:
        dict: Dictionary with tuples of opponents as keys and dictionaries with results and their probabilities as values
    """

    #Empty dictionary to store match data
    match_data = {}

    for group, teams in groups.items(): #Loop through each group

        matches = fixtures(teams) #Generate matches for the group

        for home_team, away_team in matches: #Loop through each match

            home_effect = (home_team == "QAT" or away_team == "QAT") #Home effect if either team is Qatar
            lam, mu = expected_goals(home_team, away_team, params, home_effect = home_effect)

            scores = []
            probs = []

            for home_goals in range(6): #Home goals from 0 to 5
                for away_goals in range(6): #Away goals from 0 to 5

                    p_home = poisson.pmf(home_goals, lam) #Probability of home team scoring home_goals given expected goals lam
                    p__away = poisson.pmf(away_goals, mu) #Probability of away team scoring away_goals given expected goals mu

                    prob = p_home * p__away #Probability of home_goals and away_goals happening in the same match

                    scores.append((home_goals, away_goals))
                    probs.append(prob)

            probs = np.array(probs)
            probs = probs / probs.sum() #Normalize probabilities to sum to 1

            #Store results and probabilites in match_data
            match_data[(home_team, away_team)] = {
                "Scores" : scores,
                "Probs" : probs
            }
    
    return match_data

##
**Simulation**
***

In [10]:
#Simulate a match by randomly selecting a scoreline from compute_matches
def simulate_match(scores, probs):

    """
    Simulate a match by randomly selecting a scoreline from compute_matches

    Parameters:
        scores (list[tuple[int, int]]): List of possible scorelines (home goals, away goals)
        probs (list[float]): List of probabilities corresponding to each scoreline

    Returns:
        tuple(int, int): Simulated scoreline (home goals, away goals)
    """

    random_score = np.random.choice(len(scores), p = probs) #Randomly select the index (from scores) of a scoreline by its probability
    home_goals, away_goals = scores[random_score] #Unpacks result at index random_score to home_goals and away_goals

    return home_goals, away_goals

In [11]:
#Simulate a full group and update table accordingly
def simulate_group(teams, match_data):

    """
    Simulate all matches in a group and update table accordingly

    Parameters:
        teams (list[str]): List of teams in the group
        match_data (dict): 

    Returns:
        dict: Updated table with results from simulated matches
    """

    group_table = table(teams) #Initialize table for group
    matches = fixtures(teams) #Generate matches for group

    for home_team, away_team in matches: #Loop through each match

        scores = match_data[(home_team, away_team)]["Scores"] #Get possible scores for this match from match_data
        probs = match_data[(home_team, away_team)]["Probs"] #Get corresponding probability

        home_goals, away_goals = simulate_match(scores, probs) #Simulate match by picking a scoreline based on its probability
        update_table(group_table, home_team, away_team, home_goals, away_goals) #Update table with new results

    return group_table

##
**Simulate 10 000 times**
***

In [12]:
#Monte Carlo simulation to estimate probabilities
def monte_carlo(groups, iterations):

    """
    Run a Monte Carlo simulation of the group stage to estimate probabilities

    Parameters:
        groups (dict): Dictionary with group names as keys and list of teams as values
        iterations (int): Number of iterations to run the simulation

    Returns:
        dict: Dictionary with team as key and statistics as values
    """

    stats = {}

    #Generate stats dictionary for each team
    for teams in groups.values(): #Loop  through each group
        for team in teams: #Loop through each team in group
            stats[team] = {
                "First place" : 0,
                "Second place" : 0,
                "Expected points" : 0,
                "Expected goal difference" : 0
            }

    match_data = compute_matches(groups) #Retrieve scores and probabilities to avoid doing for every iteration

    #Monte carlo simulation
    for _ in range(iterations):
        for group, teams in groups.items():

            group_table = simulate_group(teams, match_data) #Simulate group
            sorted_table = sort_table(group_table) #Sort table

            #Retrieve first and second place
            first_place = sorted_table[0][0]
            second_place = sorted_table[1][0]

            #Update first and second place stats
            stats[first_place]["First place"] += 1
            stats[second_place]["Second place"] += 1
            
            #Update points statistics for every team
            for team, team_stats in group_table.items():

                stats[team]["Expected points"] += team_stats["Points"]
                stats[team]["Expected goal difference"] += team_stats["Goal Difference"]

        results = {} #Dictionary for final results

        #Store calculated statistics for every team
        for team, team_stats in stats.items():

            #Amount of times it occurs / iterations to get probabilities
            results[team] = {
                "First place probability" : team_stats["First place"] / iterations,
                "Qualifying probability" : (team_stats["First place"] + team_stats["Second place"]) / iterations, #First and second place qualify
                "Expected points" : team_stats["Expected points"] / iterations,
                "Expected goal difference" : team_stats["Expected goal difference"] / iterations
            }

    return results

In [13]:
#Running simulation 10 000 times
results = monte_carlo(groups, iterations = 10000)

In [14]:
for group, teams in groups.items():
    print("\n" + " " * 65 + f"Group {group}")
    print("-" * 140 + "\n")


    for team in teams:
        print(f"{team}: {results[team]}")


                                                                 Group A
--------------------------------------------------------------------------------------------------------------------------------------------

NET: {'First place probability': 0.7023, 'Qualifying probability': 0.9145, 'Expected points': 6.8883, 'Expected goal difference': 4.3455}
SEN: {'First place probability': 0.0813, 'Qualifying probability': 0.3337, 'Expected points': 3.3611, 'Expected goal difference': -1.0775}
ECU: {'First place probability': 0.1984, 'Qualifying probability': 0.6458, 'Expected points': 4.6914, 'Expected goal difference': 0.939}
QAT: {'First place probability': 0.018, 'Qualifying probability': 0.106, 'Expected points': 1.8392, 'Expected goal difference': -4.207}

                                                                 Group B
--------------------------------------------------------------------------------------------------------------------------------------------

ENG: {'First place